# Supervised Training TSCP (Fase 2 + Fase 3)

Notebook ini berisi arsitektur TSCP (Two Stage CopyNet) dan proses training
supervised-nya, dibangun berdasarkan paper Sequicity (Lei 2018).
- Fase 2 = arsitektur (Encoder, Decoder Stage 1, Decoder Stage 2)
- Fase 3 = training loop (supervised learning)

Data dan vocabulary diambil dari hasil preprocessing yang sudah dibuat di tahap sebelumnya.

## Persiapan

Siapkan dulu environment-nya. Cari folder root proyek supaya notebook bisa
dijalankan dari mana saja, lalu masukkan ke sys.path sebelum memanggil modul
yang ada di folder src.

In [1]:
# cari root proyek lalu masukkan ke sys.path sebelum memanggil modul di src
import sys, os
from pathlib import Path

base = Path(os.getcwd())
ROOT = base
for pp in [base, base.parent, base.parent.parent]:
    if (pp / "data").exists():
        ROOT = pp
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import src.config as config
from src.preprocess import prepare_data, tokenize, tokens_to_indices
from src.dataset import get_dataloader
from src.utils import get_kt_from_bspan, parse_bspan

DEVICE = "cpu"  # laptop pakai AMD, jadi training di CPU
print("Root :", ROOT)
print("Device:", DEVICE)


Root : /home/benjaminasweep/Downloads/Kampus/Semester 6/Penambangan Teks/TUBES/Restaurant-Assistant-Chatbot
Device: cpu


In [2]:
# penyesuaian hyperparameter
config.EMBED_SIZE = 50
config.HIDDEN_SIZE = 50
config.SL_LEARNING_RATE = 0.003
config.SL_EPOCHS = 100
config.SL_PATIENCE = 10
config.SL_CLIP_GRAD = 5.0
config.DROPOUT = 0.0
config.SL_BATCH_SIZE = 32


## 1. Siapkan Data

Panggil preprocessing yang sudah ada supaya langsung mendapatkan data latih,
data uji, serta vocabulary yang dibutuhkan.

In [3]:
# panggil preprocessing yang sudah ada, langsung dapet data + vocab
data = prepare_data()
word2idx = data["word2idx"]
idx2word = data["idx2word"]
database = data["database"]

print(f"Train: {len(data['train'])} | Dev: {len(data['dev'])} | Test: {len(data['test'])}")
print("Vocab size:", len(word2idx))


Data split -> Train: 405, Dev: 135, Test: 136
Samples -> Train: 1635, Dev: 553, Test: 556
Vocabulary size: 755 (|V| di paper: ~800)
Train: 1635 | Dev: 553 | Test: 556
Vocab size: 755


## 2. Arsitektur Model (Fase 2)

Bagian ini berisi arsitektur TSCP yang ditulis lengkap di notebook.
Terdiri dari Encoder, Decoder Stage 1 (belief span), Decoder Stage 2 (response),
dan fungsi untuk menggabungkan peluang generate dengan copy.

In [4]:
# Encoder: membaca input lalu menghasilkan hidden states
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers=1, dropout=0.0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=0)
        self.gru = nn.GRU(
            embed_size, hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=False,
        )
        self.hidden_size = hidden_size

    def forward(self, input_seq, input_lengths=None):
        embedded = self.embedding(input_seq)

        if input_lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                embedded, input_lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            outputs, hidden = self.gru(packed)
            outputs, _ = nn.utils.rnn.pad_packed_sequence(outputs, batch_first=True)
        else:
            outputs, hidden = self.gru(embedded)

        return outputs, hidden


In [5]:
# Decoder Stage 1: menghasilkan belief span dari encoder outputs
# pakai attention (Eq. 1-3) untuk generate, dan copy (Eq. 5-6) dari input X
class DecoderStage1(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers=1, dropout=0.0):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.embed_size = embed_size

        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=0)
        self.gru = nn.GRU(
            embed_size, hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        # attention untuk generate (Eq. 1-3)
        self.W1 = nn.Linear(hidden_size, hidden_size, bias=False)
        self.W2 = nn.Linear(hidden_size, hidden_size, bias=False)
        self.v = nn.Linear(hidden_size, 1, bias=False)
        self.output_proj = nn.Linear(hidden_size * 2, vocab_size)

        # copy mechanism (Eq. 5-6)
        self.W_copy = nn.Linear(hidden_size, hidden_size, bias=False)

    def forward_step(self, dec_input, hidden, encoder_outputs, src_mask=None):
        embedded = self.embedding(dec_input)
        gru_out, new_hidden = self.gru(embedded, hidden)
        h_dec = gru_out.squeeze(1)

        # attention (Eq. 1-2)
        enc_proj = self.W1(encoder_outputs)
        dec_proj = self.W2(h_dec).unsqueeze(1).expand_as(enc_proj)
        energy = self.v(torch.tanh(enc_proj + dec_proj)).squeeze(2)
        if src_mask is not None:
            energy = energy.masked_fill(~src_mask, -1e9)
        attn_weights = F.softmax(energy, dim=1)

        context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs).squeeze(1)

        # generate logits (Eq. 3)
        concat = torch.cat([context, h_dec], dim=1)
        gen_logits = self.output_proj(concat)

        # copy scores (Eq. 5-6)
        copy_proj = torch.sigmoid(self.W_copy(encoder_outputs))
        copy_score = torch.bmm(copy_proj, h_dec.unsqueeze(2)).squeeze(2)

        return gen_logits, copy_score, new_hidden

    def forward(self, dec_input_seq, hidden, encoder_outputs, source_indices_ext, ext_vocab_size):
        batch_size, tgt_len = dec_input_seq.size()
        src_mask = source_indices_ext != 0
        all_log_probs = []

        for t in range(tgt_len):
            dec_input = dec_input_seq[:, t].unsqueeze(1)
            gen_logits, copy_score, hidden = self.forward_step(
                dec_input, hidden, encoder_outputs, src_mask
            )
            combined = _combine_copy_probs(
                gen_logits, copy_score, source_indices_ext,
                ext_vocab_size, self.vocab_size, src_mask
            )
            all_log_probs.append(combined)

        all_log_probs = torch.stack(all_log_probs, dim=1)
        return all_log_probs, hidden


In [6]:
# Decoder Stage 2: menghasilkan response
# bedanya: init hidden dari stage 1, attention/copy ke B_t, input di-concat dgn k_t
class DecoderStage2(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, kb_size=3, num_layers=1, dropout=0.0):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.embed_size = embed_size

        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=0)
        self.gru = nn.GRU(
            embed_size + kb_size, hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.W1 = nn.Linear(hidden_size, hidden_size, bias=False)
        self.W2 = nn.Linear(hidden_size, hidden_size, bias=False)
        self.v = nn.Linear(hidden_size, 1, bias=False)
        self.output_proj = nn.Linear(hidden_size * 2, vocab_size)

        self.W_copy = nn.Linear(hidden_size, hidden_size, bias=False)

    def forward_step(self, dec_input, hidden, bspan_outputs, kt, src_mask=None):
        embedded = self.embedding(dec_input)

        kt_expanded = kt.unsqueeze(1)
        gru_input = torch.cat([embedded, kt_expanded], dim=2)

        gru_out, new_hidden = self.gru(gru_input, hidden)
        h_dec = gru_out.squeeze(1)

        # attention ke bspan
        bspan_proj = self.W1(bspan_outputs)
        dec_proj = self.W2(h_dec).unsqueeze(1).expand_as(bspan_proj)
        energy = self.v(torch.tanh(bspan_proj + dec_proj)).squeeze(2)
        if src_mask is not None:
            energy = energy.masked_fill(~src_mask, -1e9)
        attn_weights = F.softmax(energy, dim=1)

        context = torch.bmm(attn_weights.unsqueeze(1), bspan_outputs).squeeze(1)

        concat = torch.cat([context, h_dec], dim=1)
        gen_logits = self.output_proj(concat)

        # copy dari bspan (Eq. 7-8)
        copy_proj = torch.sigmoid(self.W_copy(bspan_outputs))
        copy_score = torch.bmm(copy_proj, h_dec.unsqueeze(2)).squeeze(2)

        return gen_logits, copy_score, new_hidden

    def forward(self, dec_input_seq, hidden, bspan_outputs, kt,
                bspan_indices_ext, ext_vocab_size):
        batch_size, tgt_len = dec_input_seq.size()
        src_mask = bspan_indices_ext != 0
        all_log_probs = []

        for t in range(tgt_len):
            dec_input = dec_input_seq[:, t].unsqueeze(1)
            gen_logits, copy_score, hidden = self.forward_step(
                dec_input, hidden, bspan_outputs, kt, src_mask
            )
            combined = _combine_copy_probs(
                gen_logits, copy_score, bspan_indices_ext,
                ext_vocab_size, self.vocab_size, src_mask
            )
            all_log_probs.append(combined)

        all_log_probs = torch.stack(all_log_probs, dim=1)
        return all_log_probs, hidden


In [7]:
# gabung peluang generate dan copy ke satu ruang vocabulary (termasuk OOV)
# lalu softmax sekali supaya total probabilitas = 1.0
def _combine_copy_probs(gen_logits, copy_score, source_indices, ext_vocab_size, vocab_size, src_mask=None):
    batch_size = gen_logits.size(0)
    device = gen_logits.device

    extended_logits = torch.full(
        (batch_size, ext_vocab_size), fill_value=-1e10, device=device
    )

    extended_logits[:, :vocab_size] = gen_logits

    if src_mask is not None:
        copy_score = copy_score.masked_fill(~src_mask, -1e10)

    extended_logits.scatter_add_(1, source_indices, copy_score)

    log_probs = F.log_softmax(extended_logits, dim=1)

    return log_probs


In [8]:
# TSCP: gabungan encoder, decoder1, dan decoder2 menjadi satu model
class TSCP(nn.Module):
    def __init__(self, vocab_size, embed_size=config.EMBED_SIZE,
                 hidden_size=config.HIDDEN_SIZE, num_layers=config.NUM_GRU_LAYERS,
                 dropout=config.DROPOUT):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size

        self.encoder = Encoder(vocab_size, embed_size, hidden_size, num_layers, dropout)
        self.decoder1 = DecoderStage1(vocab_size, embed_size, hidden_size, num_layers, dropout)
        self.decoder2 = DecoderStage2(vocab_size, embed_size, hidden_size,
                                       kb_size=config.KB_INDICATOR_SIZE,
                                       num_layers=num_layers, dropout=dropout)

    def forward(self, input_seq, input_lengths,
                bspan_input, bspan_target,
                response_input, response_target,
                input_tokens_batch, bspan_tokens_batch,
                kt_batch, word2idx):
        device = input_seq.device
        batch_size = input_seq.size(0)

        idx2word = getattr(self, "_idx2word_cache", None)
        if idx2word is None or len(idx2word) != len(word2idx):
            idx2word = {v: k for k, v in word2idx.items()}
            self._idx2word_cache = idx2word

        # 1. encode
        encoder_outputs, encoder_hidden = self.encoder(input_seq, input_lengths)

        # 2. build extended vocab stage 1
        max_ext_vocab_s1 = self.vocab_size
        all_input_ext = []
        all_bspan_target_ext = []

        for i in range(batch_size):
            tokens = input_tokens_batch[i]
            ext_size, oov_tokens, input_ext_idx = self._build_copy_map(tokens, word2idx)
            max_ext_vocab_s1 = max(max_ext_vocab_s1, ext_size)

            src_len = input_seq.size(1)
            padded = input_ext_idx + [0] * (src_len - len(input_ext_idx))
            all_input_ext.append(padded[:src_len])

            bspan_tgt_tokens = self._indices_to_tokens(bspan_target[i], idx2word)
            bspan_tgt_ext = self._target_to_ext(bspan_tgt_tokens, word2idx, oov_tokens)
            bspan_len = bspan_target.size(1)
            bspan_tgt_ext = bspan_tgt_ext + [0] * (bspan_len - len(bspan_tgt_ext))
            all_bspan_target_ext.append(bspan_tgt_ext[:bspan_len])

        input_ext_tensor = torch.tensor(all_input_ext, dtype=torch.long, device=device)
        bspan_target_ext = torch.tensor(all_bspan_target_ext, dtype=torch.long, device=device)

        # 3. decode bspan (stage 1)
        bspan_log_probs, bspan_final_hidden = self.decoder1(
            bspan_input, encoder_hidden, encoder_outputs,
            input_ext_tensor, max_ext_vocab_s1
        )
        bspan_loss = self._compute_loss(bspan_log_probs, bspan_target_ext, max_ext_vocab_s1)

        # 4. bspan hidden states untuk stage 2
        bspan_embedded = self.decoder1.embedding(bspan_input)
        bspan_outputs, _ = self.decoder1.gru(bspan_embedded, encoder_hidden)

        # 5. build extended vocab stage 2
        max_ext_vocab_s2 = self.vocab_size
        all_bspan_ext = []
        all_resp_target_ext = []

        for i in range(batch_size):
            tokens = bspan_tokens_batch[i]
            ext_size, oov_tokens, bspan_ext_idx = self._build_copy_map(tokens, word2idx)
            max_ext_vocab_s2 = max(max_ext_vocab_s2, ext_size)

            bspan_len = bspan_input.size(1)
            padded = bspan_ext_idx + [0] * (bspan_len - len(bspan_ext_idx))
            all_bspan_ext.append(padded[:bspan_len])

            resp_tgt_tokens = self._indices_to_tokens(response_target[i], idx2word)
            resp_tgt_ext = self._target_to_ext(resp_tgt_tokens, word2idx, oov_tokens)
            resp_len = response_target.size(1)
            resp_tgt_ext = resp_tgt_ext + [0] * (resp_len - len(resp_tgt_ext))
            all_resp_target_ext.append(resp_tgt_ext[:resp_len])

        bspan_ext_tensor = torch.tensor(all_bspan_ext, dtype=torch.long, device=device)
        resp_target_ext = torch.tensor(all_resp_target_ext, dtype=torch.long, device=device)

        # 6. decode response (stage 2)
        response_log_probs, _ = self.decoder2(
            response_input, bspan_final_hidden, bspan_outputs, kt_batch,
            bspan_ext_tensor, max_ext_vocab_s2
        )
        response_loss = self._compute_loss(response_log_probs, resp_target_ext, max_ext_vocab_s2)

        total_loss = bspan_loss + response_loss
        return total_loss, bspan_loss, response_loss

    def _compute_loss(self, log_probs, targets, ext_vocab_size):
        batch_size, seq_len, _ = log_probs.size()

        log_probs_flat = log_probs.view(-1, ext_vocab_size)
        targets_flat = targets.view(-1)

        non_pad_mask = targets_flat.ne(0)
        targets_clamped = targets_flat.clamp(0, ext_vocab_size - 1)
        nll = -log_probs_flat.gather(1, targets_clamped.unsqueeze(1)).squeeze(1)

        nll = nll * non_pad_mask.float()
        loss = nll.sum() / non_pad_mask.float().sum().clamp(min=1)

        return loss

    def _build_copy_map(self, tokens, word2idx):
        vocab_size = len(word2idx)
        oov_tokens = []
        oov_map = {}
        ext_indices = []

        for token in tokens:
            if token in word2idx:
                ext_indices.append(word2idx[token])
            else:
                if token not in oov_map:
                    oov_map[token] = vocab_size + len(oov_tokens)
                    oov_tokens.append(token)
                ext_indices.append(oov_map[token])

        ext_size = vocab_size + len(oov_tokens)
        return ext_size, oov_tokens, ext_indices

    def _target_to_ext(self, target_tokens, word2idx, oov_tokens):
        vocab_size = len(word2idx)
        unk_idx = word2idx[config.UNK_TOKEN]
        oov_map = {tok: vocab_size + i for i, tok in enumerate(oov_tokens)}

        indices = []
        for token in target_tokens:
            if token in word2idx:
                indices.append(word2idx[token])
            elif token in oov_map:
                indices.append(oov_map[token])
            else:
                indices.append(unk_idx)
        return indices

    def _indices_to_tokens(self, index_tensor, idx2word):
        tokens = []
        for idx in index_tensor:
            idx_val = idx.item() if isinstance(idx, torch.Tensor) else idx
            tokens.append(idx2word.get(idx_val, config.UNK_TOKEN))
        return tokens


## 3. Training (Fase 3)

Loop training di bawah ini menjalankan proses forward, menghitung loss,
lalu backward dan memperbarui bobot. Diulang setiap epoch sampai early
stopping aktif.

In [9]:
# hitung k_t (info database) untuk tiap sample di dalam batch
def compute_kt_for_batch(batch, database, word2idx):
    kt_list = []
    for i in range(len(batch["bspan_tokens"])):
        bspan_tokens = batch["bspan_tokens"][i]
        bspan_text = " ".join(bspan_tokens)
        kt = get_kt_from_bspan(bspan_text, database)
        kt_list.append(kt)
    return torch.stack(kt_list)


In [10]:
# satu epoch training: lewat semua batch, tiap batch forward lalu backward
def train_epoch(model, dataloader, optimizer, database, word2idx, device, clip_grad):
    model.train()
    total_loss = 0.0
    total_bspan_loss = 0.0
    total_response_loss = 0.0
    num_batches = 0

    for batch in dataloader:
        input_seq = batch["input_padded"].to(device)
        input_lengths = batch["input_lengths"]
        bspan_input = batch["bspan_input"].to(device)
        bspan_target = batch["bspan_target"].to(device)
        response_input = batch["response_input"].to(device)
        response_target = batch["response_target"].to(device)

        kt_batch = compute_kt_for_batch(batch, database, word2idx).to(device)

        loss, bspan_loss, response_loss = model(
            input_seq, input_lengths,
            bspan_input, bspan_target,
            response_input, response_target,
            batch["input_tokens"], batch["bspan_tokens"],
            kt_batch, word2idx,
        )

        optimizer.zero_grad()
        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
        optimizer.step()

        total_loss += loss.item()
        total_bspan_loss += bspan_loss.item()
        total_response_loss += response_loss.item()
        num_batches += 1

    avg_loss = total_loss / max(num_batches, 1)
    avg_bspan = total_bspan_loss / max(num_batches, 1)
    avg_response = total_response_loss / max(num_batches, 1)
    return avg_loss, avg_bspan, avg_response


In [11]:
# evaluasi pada dev set untuk mengecek early stopping
def evaluate_epoch(model, dataloader, database, word2idx, device):
    model.eval()
    total_loss = 0.0
    num_batches = 0

    with torch.no_grad():
        for batch in dataloader:
            input_seq = batch["input_padded"].to(device)
            input_lengths = batch["input_lengths"]
            bspan_input = batch["bspan_input"].to(device)
            bspan_target = batch["bspan_target"].to(device)
            response_input = batch["response_input"].to(device)
            response_target = batch["response_target"].to(device)

            kt_batch = compute_kt_for_batch(batch, database, word2idx).to(device)

            loss, _, _ = model(
                input_seq, input_lengths,
                bspan_input, bspan_target,
                response_input, response_target,
                batch["input_tokens"], batch["bspan_tokens"],
                kt_batch, word2idx,
            )

            total_loss += loss.item()
            num_batches += 1

    return total_loss / max(num_batches, 1)


In [12]:
# training loop utama: jalankan epoch, simpan model terbaik, lalu early stopping
def train_supervised(data, device="cpu"):
    word2idx = data["word2idx"]
    database = data["database"]
    vocab_size = len(word2idx)

    print(f"\n{'='*60}")
    print(f"SUPERVISED TRAINING")
    print(f"{'='*60}")
    print(f"Vocab size: {vocab_size}")
    print(f"Device: {device}")
    print(f"Batch size: {config.SL_BATCH_SIZE}")
    print(f"Learning rate: {config.SL_LEARNING_RATE}")
    print(f"Hidden/Embed size: {config.HIDDEN_SIZE}/{config.EMBED_SIZE}")
    print(f"Max epochs: {config.SL_EPOCHS}")
    print(f"Early stopping patience: {config.SL_PATIENCE}")

    # dataloader
    train_loader = get_dataloader(data["train"], word2idx, config.SL_BATCH_SIZE, shuffle=True)
    dev_loader = get_dataloader(data["dev"], word2idx, config.SL_BATCH_SIZE, shuffle=False)

    # model (arsitektur yang dibuat di atas)
    model = TSCP(vocab_size).to(device)
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")

    # optimizer
    optimizer = optim.Adam(model.parameters(), lr=config.SL_LEARNING_RATE)

    # early stopping
    best_dev_loss = float("inf")
    patience_counter = 0
    best_model_state = None

    os.makedirs(config.SAVE_DIR, exist_ok=True)

    print(f"\nStarting training...\n")

    for epoch in range(1, config.SL_EPOCHS + 1):
        start_time = time.time()

        train_loss, train_bspan, train_resp = train_epoch(
            model, train_loader, optimizer, database, word2idx, device, config.SL_CLIP_GRAD
        )

        dev_loss = evaluate_epoch(model, dev_loader, database, word2idx, device)

        elapsed = time.time() - start_time

        print(
            f"Epoch {epoch:3d}/{config.SL_EPOCHS} | "
            f"Train Loss: {train_loss:.4f} (B:{train_bspan:.4f} R:{train_resp:.4f}) | "
            f"Dev Loss: {dev_loss:.4f} | "
            f"Time: {elapsed:.1f}s"
        )

        if dev_loss < best_dev_loss:
            best_dev_loss = dev_loss
            patience_counter = 0
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            checkpoint_path = os.path.join(config.SAVE_DIR, "tscp_supervised_best.pt")
            torch.save({
                "epoch": epoch,
                "model_state_dict": best_model_state,
                "optimizer_state_dict": optimizer.state_dict(),
                "dev_loss": best_dev_loss,
                "word2idx": word2idx,
                "vocab_size": vocab_size,
            }, checkpoint_path)
            print(f"  -> Best model saved (dev_loss={best_dev_loss:.4f})")
        else:
            patience_counter += 1
            if patience_counter >= config.SL_PATIENCE:
                print(f"\nEarly stopping at epoch {epoch} (patience={config.SL_PATIENCE})")
                break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"\nRestored best model (dev_loss={best_dev_loss:.4f})")

    return model


## 4. Jalankan Training

Jalankan proses training supervised untuk melatih model.

In [13]:
# jalankan training supervised
import time

model = train_supervised(data, device=DEVICE)
print("Training selesai.")



SUPERVISED TRAINING
Vocab size: 755
Device: cpu
Batch size: 32
Learning rate: 0.003
Hidden/Embed size: 50/50
Max epochs: 100
Early stopping patience: 10
Total parameters: 327,210
Trainable parameters: 327,210

Starting training...

Epoch   1/100 | Train Loss: 6.4846 (B:2.2129 R:4.2717) | Dev Loss: 3.6321 | Time: 4.9s
  -> Best model saved (dev_loss=3.6321)
Epoch   2/100 | Train Loss: 2.8926 (B:0.3670 R:2.5256) | Dev Loss: 2.4821 | Time: 4.6s
  -> Best model saved (dev_loss=2.4821)
Epoch   3/100 | Train Loss: 2.1953 (B:0.2432 R:1.9521) | Dev Loss: 2.1058 | Time: 4.5s
  -> Best model saved (dev_loss=2.1058)
Epoch   4/100 | Train Loss: 1.8876 (B:0.2030 R:1.6846) | Dev Loss: 1.9235 | Time: 4.7s
  -> Best model saved (dev_loss=1.9235)
Epoch   5/100 | Train Loss: 1.7080 (B:0.1718 R:1.5362) | Dev Loss: 1.8088 | Time: 4.6s
  -> Best model saved (dev_loss=1.8088)
Epoch   6/100 | Train Loss: 1.5958 (B:0.1528 R:1.4430) | Dev Loss: 1.7483 | Time: 4.6s
  -> Best model saved (dev_loss=1.7483)
Epoch

## 5. Evaluasi Test Set

Setelah training selesai, model diuji pada test set menggunakan 3 metrik:
- BLEU: menilai kualitas bahasa pada jawaban
- Entity Match Rate: mengecek apakah belief span menghasilkan constraint yang benar
- Success F1: mengecek apakah jawaban memuat informasi yang diminta user

Proses prediksi menggunakan greedy decoding (mengambil token dengan peluang
terbesar pada setiap langkah).

In [14]:
# import tambahan untuk evaluasi dan membangun extended vocab per sample
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from src.utils import (
    parse_bspan, search_kb, lexicalize_response, resolve_inconsistent_bspan,
)

# bangun extended vocab untuk satu sample (buat greedy decode)
def build_source_ext_tensor(tokens, word2idx, device):
    vocab_size = len(word2idx)
    oov_tokens = []
    oov_map = {}
    ext_indices = []
    for token in tokens:
        if token in word2idx:
            ext_indices.append(word2idx[token])
        else:
            if token not in oov_map:
                oov_map[token] = vocab_size + len(oov_tokens)
                oov_tokens.append(token)
            ext_indices.append(oov_map[token])
    ext_vocab_size = vocab_size + len(oov_tokens)
    source_tensor = torch.tensor([ext_indices], dtype=torch.long, device=device)
    return ext_vocab_size, oov_tokens, source_tensor


In [15]:
# greedy decode Stage 1 (belief span)
def greedy_decode_bspan(model, encoder_outputs, encoder_hidden, word2idx, idx2word,
                        input_tokens, device):
    sos_idx = word2idx[config.SOS_TOKEN]
    eos_idx = word2idx[config.EOS_TOKEN]
    vocab_size = len(word2idx)

    ext_vocab_size, oov_tokens, source_ext = build_source_ext_tensor(
        input_tokens, word2idx, device
    )

    hidden = encoder_hidden
    dec_input = torch.tensor([[sos_idx]], dtype=torch.long, device=device)
    generated_tokens = []

    for _ in range(config.MAX_DECODE_LEN_BSPAN):
        gen_logits, copy_score, hidden = model.decoder1.forward_step(
            dec_input, hidden, encoder_outputs
        )
        log_probs = _combine_copy_probs(
            gen_logits, copy_score, source_ext, ext_vocab_size, vocab_size
        )
        best_idx = log_probs.squeeze(0).argmax().item()

        if best_idx == eos_idx:
            break

        if best_idx < vocab_size:
            token = idx2word.get(best_idx, config.UNK_TOKEN)
        else:
            oov_offset = best_idx - vocab_size
            token = oov_tokens[oov_offset] if oov_offset < len(oov_tokens) else config.UNK_TOKEN

        generated_tokens.append(token)
        next_idx = best_idx if best_idx < vocab_size else word2idx[config.UNK_TOKEN]
        dec_input = torch.tensor([[next_idx]], dtype=torch.long, device=device)

    return generated_tokens


In [16]:
# greedy decode Stage 2 (response)
def greedy_decode_response(model, bspan_outputs, bspan_final_hidden,
                           kt, word2idx, idx2word, bspan_tokens, device):
    sos_idx = word2idx[config.SOS_TOKEN]
    eos_idx = word2idx[config.EOS_TOKEN]
    vocab_size = len(word2idx)

    ext_vocab_size, oov_tokens, bspan_ext = build_source_ext_tensor(
        bspan_tokens, word2idx, device
    )

    hidden = bspan_final_hidden
    dec_input = torch.tensor([[sos_idx]], dtype=torch.long, device=device)
    kt_tensor = kt.unsqueeze(0).to(device)
    generated_tokens = []

    for _ in range(config.MAX_DECODE_LEN_RESPONSE):
        gen_logits, copy_score, hidden = model.decoder2.forward_step(
            dec_input, hidden, bspan_outputs, kt_tensor
        )
        log_probs = _combine_copy_probs(
            gen_logits, copy_score, bspan_ext, ext_vocab_size, vocab_size
        )
        best_idx = log_probs.squeeze(0).argmax().item()

        if best_idx == eos_idx:
            break

        if best_idx < vocab_size:
            token = idx2word.get(best_idx, config.UNK_TOKEN)
        else:
            oov_offset = best_idx - vocab_size
            token = oov_tokens[oov_offset] if oov_offset < len(oov_tokens) else config.UNK_TOKEN

        generated_tokens.append(token)
        next_idx = best_idx if best_idx < vocab_size else word2idx[config.UNK_TOKEN]
        dec_input = torch.tensor([[next_idx]], dtype=torch.long, device=device)

    return generated_tokens


In [17]:
# hasilkan bspan dan response untuk satu sample pada test set
def generate_for_sample(model, sample, word2idx, idx2word, database, device):
    model.eval()
    with torch.no_grad():
        input_tokens = tokenize(sample["input"])
        input_indices = tokens_to_indices(input_tokens, word2idx)
        input_tensor = torch.tensor([input_indices], dtype=torch.long, device=device)
        input_lengths = torch.tensor([len(input_indices)])

        # 1. encode
        encoder_outputs, encoder_hidden = model.encoder(input_tensor, input_lengths)

        # 2. decode bspan
        pred_bspan_tokens = greedy_decode_bspan(
            model, encoder_outputs, encoder_hidden, word2idx, idx2word,
            input_tokens, device
        )
        pred_bspan_text = " ".join(pred_bspan_tokens)

        # post-processing bspan
        pred_bspan_text = resolve_inconsistent_bspan(pred_bspan_text, database)
        pred_bspan_tokens = tokenize(pred_bspan_text)

        # 3. KB search
        kb_matches, kt = search_kb(pred_bspan_text, database)

        # 4. bspan hidden states untuk stage 2
        bspan_indices = tokens_to_indices(pred_bspan_tokens, word2idx)
        sos_idx = word2idx[config.SOS_TOKEN]
        bspan_input_tensor = torch.tensor(
            [[sos_idx] + bspan_indices], dtype=torch.long, device=device
        )
        bspan_embedded = model.decoder1.embedding(bspan_input_tensor)
        bspan_outputs, bspan_final_hidden = model.decoder1.gru(
            bspan_embedded, encoder_hidden
        )

        # 5. decode response
        full_bspan_tokens = [config.SOS_TOKEN] + pred_bspan_tokens
        pred_response_tokens = greedy_decode_response(
            model, bspan_outputs, bspan_final_hidden,
            kt, word2idx, idx2word, full_bspan_tokens, device
        )
        pred_response_text = " ".join(pred_response_tokens)

    return pred_bspan_text, pred_response_text, kb_matches


In [18]:
# metrik 1: BLEU (kualitas bahasa response)
def compute_bleu(references, hypotheses):
    refs = [[ref] for ref in references]
    smooth = SmoothingFunction().method1

    filtered_refs = []
    filtered_hyps = []
    for ref, hyp in zip(refs, hypotheses):
        if len(hyp) > 0:
            filtered_refs.append(ref)
            filtered_hyps.append(hyp)

    if not filtered_hyps:
        return 0.0
    return corpus_bleu(filtered_refs, filtered_hyps, smoothing_function=smooth)


# metrik 2: Entity Match Rate (bspan constraint benar atau tidak)
def compute_entity_match_rate(pred_bspans, gold_bspans, database):
    matches = 0
    total = 0
    for pred_b, gold_b in zip(pred_bspans, gold_bspans):
        pred_inf, _ = parse_bspan(pred_b)
        gold_inf, _ = parse_bspan(gold_b)
        if set(v.lower() for v in pred_inf) == set(v.lower() for v in gold_inf):
            matches += 1
        total += 1
    return matches / max(total, 1)


# metrik 3: Success F1 (response memuat request slot yang diminta)
def compute_success_f1(pred_responses, gold_bspans):
    slot_to_placeholder = {
        "address": "address_slot", "phone": "phone_slot", "postcode": "postcode_slot",
        "food": "food_slot", "area": "area_slot", "pricerange": "pricerange_slot",
        "name": "name_slot",
    }
    all_precision = []
    all_recall = []

    for pred_resp, gold_b in zip(pred_responses, gold_bspans):
        _, requestable = parse_bspan(gold_b)
        if not requestable:
            continue

        expected = set()
        for req in requestable:
            ph = slot_to_placeholder.get(req.lower())
            if ph:
                expected.add(ph)
        if not expected:
            continue

        pred_lower = pred_resp.lower()
        found = {ph for ph in expected if ph in pred_lower}
        all_ph_in_resp = {ph for ph in slot_to_placeholder.values() if ph in pred_lower}

        recall = len(found) / len(expected) if expected else 0.0
        if all_ph_in_resp:
            precision = len(found.intersection(all_ph_in_resp)) / len(all_ph_in_resp)
        else:
            precision = 0.0

        all_precision.append(precision)
        all_recall.append(recall)

    if not all_precision:
        return 0.0
    avg_p = sum(all_precision) / len(all_precision)
    avg_r = sum(all_recall) / len(all_recall)
    if avg_p + avg_r == 0:
        return 0.0
    return 2 * avg_p * avg_r / (avg_p + avg_r)


In [19]:
# jalankan evaluasi di seluruh test set
def evaluate_model(model, test_samples, word2idx, idx2word, database, device="cpu"):
    print(f"\n{'='*60}")
    print(f"EVALUASI TEST SET ({len(test_samples)} samples)")
    print(f"{'='*60}")

    model.eval()
    all_pred_bspans = []
    all_gold_bspans = []
    all_pred_responses = []
    all_gold_responses = []

    for i, sample in enumerate(test_samples):
        pred_bspan, pred_response, kb_matches = generate_for_sample(
            model, sample, word2idx, idx2word, database, device
        )
        all_pred_bspans.append(pred_bspan)
        all_gold_bspans.append(sample["target_bspan"])
        all_pred_responses.append(pred_response)
        all_gold_responses.append(sample["target_response"])

        # tampilkan 5 contoh pertama
        if i < 5:
            print(f"\n--- Sample {i} ---")
            print(f"  Input        : {sample['input'][:80]}...")
            print(f"  Gold Bspan   : {sample['target_bspan']}")
            print(f"  Pred Bspan   : {pred_bspan}")
            print(f"  Gold Response: {sample['target_response'][:80]}...")
            print(f"  Pred Response: {pred_response[:80]}...")

    # hitung metrik
    ref_tok = [tokenize(r) for r in all_gold_responses]
    hyp_tok = [tokenize(r) for r in all_pred_responses]
    bleu = compute_bleu(ref_tok, hyp_tok)
    entity_match = compute_entity_match_rate(all_pred_bspans, all_gold_bspans, database)
    success_f1 = compute_success_f1(all_pred_responses, all_gold_bspans)

    print(f"\n{'='*60}")
    print(f"HASIL EVALUASI")
    print(f"{'='*60}")
    print(f"  BLEU              : {bleu:.4f}")
    print(f"  Entity Match Rate : {entity_match:.4f}")
    print(f"  Success F1        : {success_f1:.4f}")
    print(f"{'='*60}")

    return {"BLEU": bleu, "Entity_Match_Rate": entity_match, "Success_F1": success_f1}


# jalankan evaluasi
results = evaluate_model(model, data["test"], word2idx, idx2word, database, device=DEVICE)



EVALUASI TEST SET (556 samples)

--- Sample 0 ---
  Input        : i'm looking for a moderately priced restaurant serving cuban food....
  Gold Bspan   : <inf> cuban ; moderate </inf> <req>  </req>
  Pred Bspan   : <inf> moderate </inf> <req>  </req>
  Gold Response: i'm sorry we do not have any cuban restaurants in the PRICERANGE_SLOT price rang...
  Pred Response: NAME_SLOT is an PRICERANGE_SLOT restaurant in the AREA_SLOT part of town . would...

--- Sample 1 ---
  Input        : <inf> cuban ; moderate </inf> <req>  </req> i'm sorry we do not have any cuban r...
  Gold Bspan   : <inf> british ; moderate </inf> <req>  </req>
  Pred Bspan   : <inf> british ; moderate </inf> <req>  </req>
  Gold Response: NAME_SLOT serves FOOD_SLOT food. would you like more information about this loca...
  Pred Response: NAME_SLOT is an PRICERANGE_SLOT FOOD_SLOT restaurant in the AREA_SLOT part of to...

--- Sample 2 ---
  Input        : <inf> british ; moderate </inf> <req>  </req> NAME_SLOT serves F

## 6. Cek Hasil

Periksa hasil akhir berupa model terbaik yang disimpan pada checkpoint.

In [20]:
# cek checkpoint hasil training
ckpt = torch.load(os.path.join(config.SAVE_DIR, "tscp_supervised_best.pt"),
                  map_location=DEVICE, weights_only=False)
print("Epoch terbaik :", ckpt["epoch"])
print(f"Dev loss      : {ckpt['dev_loss']:.4f}")
print("Vocab size    :", ckpt["vocab_size"])


Epoch terbaik : 15
Dev loss      : 1.5776
Vocab size    : 755
